In [135]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import plotly.express as px
import os
import koreanize_matplotlib
import re
import folium
import json
import requests
import warnings
from folium.plugins import HeatMap
from scipy.stats import mannwhitneyu
from scipy.stats import pearsonr

# 유무임 승하차 인원 데이터 전처리

In [136]:
s2023_df = pd.read_csv('../../data/raw/서울시 지하철 호선별 역별 유_무임 승하차 인원 정보_2023.csv')

In [137]:
s2023_df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7386 entries, 0 to 7385
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   사용월     7386 non-null   int64 
 1   호선명     7386 non-null   object
 2   지하철역    7386 non-null   object
 3   유임승차인원  7386 non-null   int64 
 4   무임승차인원  7386 non-null   int64 
 5   유임하차인원  7386 non-null   int64 
 6   무임하차인원  7386 non-null   int64 
 7   작업일자    7386 non-null   int64 
dtypes: int64(6), object(2)
memory usage: 461.8+ KB


In [138]:
print(s2023_df.isnull().sum())
print()
print(f'중복행: {s2023_df.duplicated().sum()}')

사용월       0
호선명       0
지하철역      0
유임승차인원    0
무임승차인원    0
유임하차인원    0
무임하차인원    0
작업일자      0
dtype: int64

중복행: 0


In [139]:
s2023_df[s2023_df.duplicated(keep=False)]

,사용월,호선명,지하철역,유임승차인원,무임승차인원,유임하차인원,무임하차인원,작업일자


In [140]:
s2024_df = pd.read_csv('../../data/raw/서울시 지하철 호선별 역별 유_무임 승하차 인원 정보_2024.csv')

In [141]:
s2024_df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   사용월     8068 non-null   int64 
 1   호선명     8068 non-null   object
 2   지하철역    8068 non-null   object
 3   유임승차인원  8068 non-null   int64 
 4   무임승차인원  8068 non-null   int64 
 5   유임하차인원  8068 non-null   int64 
 6   무임하차인원  8068 non-null   int64 
 7   작업일자    8068 non-null   int64 
dtypes: int64(6), object(2)
memory usage: 504.4+ KB


In [142]:
print(s2024_df.isnull().sum())
print()
print(f'중복행: {s2024_df.duplicated().sum()}')

사용월       0
호선명       0
지하철역      0
유임승차인원    0
무임승차인원    0
유임하차인원    0
무임하차인원    0
작업일자      0
dtype: int64

중복행: 620


In [143]:
s2024_df[s2024_df.duplicated(keep=False)]

,사용월,호선명,지하철역,유임승차인원,무임승차인원,유임하차인원,무임하차인원,작업일자
6209,202402,1호선,동묘앞,144619,142038,155082,144687,20240303
6210,202402,1호선,청량리(서울시립대입구),350437,266447,340983,268232,20240303
6211,202402,1호선,제기동,215365,254164,207846,271449,20240303
6212,202402,1호선,신설동,257411,110367,248804,107289,20240303
6213,202402,1호선,동대문,224713,113712,225306,113175,20240303
...,...,...,...,...,...,...,...,...
7444,202402,중앙선,상봉,119677,37842,117749,36705,20240303
7445,202402,중앙선,중랑,101785,45693,97927,44969,20240303
7446,202402,중앙선,회기,540835,111210,519096,109273,20240303
7447,202402,중앙선,지평,680,837,737,836,20240303


In [144]:
s2024_df[s2024_df['사용월'] == 202402].duplicated().sum()

np.int64(620)

In [145]:
s2024_df = s2024_df.drop_duplicates()

print(f"중복 제거 후 전체 행 개수: {len(s2024_df)}")
print(f"남은 중복 행 개수: {s2024_df.duplicated().sum()}")

중복 제거 후 전체 행 개수: 7448
남은 중복 행 개수: 0


In [146]:
s2025_df = pd.read_csv('../../data/raw/서울시 지하철 호선별 역별 유_무임 승하차 인원 정보_2025.csv')

In [147]:
s2025_df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7456 entries, 0 to 7455
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   사용월     7456 non-null   int64 
 1   호선명     7456 non-null   object
 2   지하철역    7456 non-null   object
 3   유임승차인원  7456 non-null   int64 
 4   무임승차인원  7456 non-null   int64 
 5   유임하차인원  7456 non-null   int64 
 6   무임하차인원  7456 non-null   int64 
 7   작업일자    7456 non-null   int64 
dtypes: int64(6), object(2)
memory usage: 466.1+ KB


In [148]:
print(s2025_df.isnull().sum())
print()
print(f'중복행: {s2025_df.duplicated().sum()}')

사용월       0
호선명       0
지하철역      0
유임승차인원    0
무임승차인원    0
유임하차인원    0
무임하차인원    0
작업일자      0
dtype: int64

중복행: 0


In [149]:
subway_user_df = pd.concat([s2023_df,s2024_df,s2025_df], ignore_index=True)
subway_user_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22290 entries, 0 to 22289
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   사용월     22290 non-null  int64 
 1   호선명     22290 non-null  object
 2   지하철역    22290 non-null  object
 3   유임승차인원  22290 non-null  int64 
 4   무임승차인원  22290 non-null  int64 
 5   유임하차인원  22290 non-null  int64 
 6   무임하차인원  22290 non-null  int64 
 7   작업일자    22290 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 1.4+ MB


In [150]:
line_list = [f'{i}호선' for i in range(1, 10)]
line_1_9_df = subway_user_df[subway_user_df['호선명'].isin(line_list)]

In [151]:
print(line_1_9_df['호선명'].unique())

['1호선' '2호선' '3호선' '4호선' '5호선' '6호선' '7호선' '8호선' '9호선']


In [152]:
# 상봉역 통일
line_1_9_df.loc[:, '지하철역'] = line_1_9_df['지하철역'].str.replace('상봉(시외버스터미널)', '상봉', regex=False)

In [153]:
# 각 역의 총 승하차인원, 무임승하차인원
line_1_9_df = line_1_9_df.copy()

line_1_9_df.loc[:, '총승차인원'] = line_1_9_df['유임승차인원'] + line_1_9_df['무임승차인원']
line_1_9_df.loc[:, '총하차인원'] = line_1_9_df['유임하차인원'] + line_1_9_df['무임하차인원']

mean_df = line_1_9_df.groupby(['호선명', '지하철역'])[['총승차인원', '무임승차인원', '총하차인원', '무임하차인원']].mean().round(0).reset_index()

print(mean_df)

     호선명       지하철역      총승차인원    무임승차인원      총하차인원    무임하차인원
0    1호선        동대문   363678.0  125523.0   356249.0  125026.0
1    1호선        동묘앞   308894.0  157888.0   316892.0  159511.0
2    1호선        서울역  1794960.0  229681.0  1701907.0  217787.0
3    1호선         시청   762366.0  107582.0   766076.0  105764.0
4    1호선        신설동   411861.0  120797.0   397175.0  117111.0
..   ...        ...        ...       ...        ...       ...
308  9호선       양천향교   289659.0   49752.0   294136.0   49308.0
309  9호선        여의도   840341.0   56004.0   804546.0   51738.0
310  9호선         염창   514529.0   80870.0   492565.0   78794.0
311  9호선         증미   219073.0   38743.0   210572.0   38823.0
312  9호선  흑석(중앙대입구)   274749.0   60214.0   279485.0   60496.0

[313 rows x 6 columns]


In [154]:
# 1~9호선에서 이제 서울에만 해당되는 데이터로 추린다.
non_seoul = [
    # 5호선 (하남)
    '미사', '하남검단산', '하남시청(덕풍?신장)', '하남풍산',
    # 7호선 (경기/인천)
    '광명사거리', '굴포천', '까치울', '부천시청', '부평구청',
    '삼산체육관', '상동', '신중동', '춘의', '철산',
    # 8호선 (성남)
    '남위례', '남한산성입구(성남법원.검찰청)', '단대오거리',
    '모란', '복정', '산성', '수진', '신흥',
]
seoul_mean_df = mean_df[~mean_df['지하철역'].isin(non_seoul)]
print(f'제외 전: {len(mean_df)}개역 → 제외 후: {len(seoul_mean_df)}개역')

제외 전: 313개역 → 제외 후: 291개역


In [155]:
# 무임승하차 비중 계산
seoul_mean_df = seoul_mean_df.copy()

seoul_mean_df['전체승하차'] = seoul_mean_df['총승차인원'] + seoul_mean_df['총하차인원']
seoul_mean_df['무임승하차'] = seoul_mean_df['무임승차인원'] + seoul_mean_df['무임하차인원']
seoul_mean_df['무임승하차비중'] = (seoul_mean_df['무임승하차'] / seoul_mean_df['전체승하차'] * 100).round(2)

print(seoul_mean_df[['호선명', '지하철역', '전체승하차', '무임승하차', '무임승하차비중']])

     호선명       지하철역      전체승하차     무임승하차  무임승하차비중
0    1호선        동대문   719927.0  250549.0    34.80
1    1호선        동묘앞   625786.0  317399.0    50.72
2    1호선        서울역  3496867.0  447468.0    12.80
3    1호선         시청  1528442.0  213346.0    13.96
4    1호선        신설동   809036.0  237908.0    29.41
..   ...        ...        ...       ...      ...
308  9호선       양천향교   583795.0   99060.0    16.97
309  9호선        여의도  1644887.0  107742.0     6.55
310  9호선         염창  1007094.0  159664.0    15.85
311  9호선         증미   429645.0   77566.0    18.05
312  9호선  흑석(중앙대입구)   554234.0  120710.0    21.78

[291 rows x 5 columns]


In [156]:
# seoul_mean_df.to_csv('subway_mean.csv', index=False, encoding='utf-8-sig')

In [157]:
# 지하철역 칼럼에서 정규식() 지우기.
seoul_mean_df['호선명'] = seoul_mean_df['호선명'].str.replace(r'\(.*?\)', '', regex=True).str.strip()

print(seoul_mean_df['호선명'].unique())

['1호선' '2호선' '3호선' '4호선' '5호선' '6호선' '7호선' '8호선' '9호선']


In [158]:
seoul_mean_df

,호선명,지하철역,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중
0,1호선,동대문,363678.0,125523.0,356249.0,125026.0,719927.0,250549.0,34.80
1,1호선,동묘앞,308894.0,157888.0,316892.0,159511.0,625786.0,317399.0,50.72
2,1호선,서울역,1794960.0,229681.0,1701907.0,217787.0,3496867.0,447468.0,12.80
3,1호선,시청,762366.0,107582.0,766076.0,105764.0,1528442.0,213346.0,13.96
4,1호선,신설동,411861.0,120797.0,397175.0,117111.0,809036.0,237908.0,29.41
...,...,...,...,...,...,...,...,...,...
308,9호선,양천향교,289659.0,49752.0,294136.0,49308.0,583795.0,99060.0,16.97
309,9호선,여의도,840341.0,56004.0,804546.0,51738.0,1644887.0,107742.0,6.55
310,9호선,염창,514529.0,80870.0,492565.0,78794.0,1007094.0,159664.0,15.85
311,9호선,증미,219073.0,38743.0,210572.0,38823.0,429645.0,77566.0,18.05


In [159]:
# seoul_mean_df.to_csv('subway_mean.csv', index=False, encoding='utf-8-sig')

In [160]:
# 상봉역이 2023년 → 상봉(시외버스터미널) (7호선) 2024년부터 → 상봉 (7호선) 으로 되어있어 2개여서  확인용
print(line_1_9_df[line_1_9_df['지하철역'].str.contains('상봉')][['호선명', '지하철역', '사용월']].sort_values('사용월'))

       호선명 지하철역     사용월
7000   7호선   상봉  202301
6407   7호선   상봉  202302
5766   7호선   상봉  202303
5172   7호선   상봉  202304
4559   7호선   상봉  202305
3952   7호선   상봉  202306
3313   7호선   상봉  202307
2716   7호선   상봉  202308
2082   7호선   상봉  202309
1476   7호선   상봉  202310
852    7호선   상봉  202311
247    7호선   상봉  202312
14461  7호선   상봉  202401
13810  7호선   상봉  202402
13200  7호선   상봉  202403
12575  7호선   상봉  202404
11959  7호선   상봉  202405
11358  7호선   상봉  202406
10716  7호선   상봉  202407
10096  7호선   상봉  202408
9493   7호선   상봉  202409
8863   7호선   상봉  202410
8253   7호선   상봉  202411
7612   7호선   상봉  202412
21915  7호선   상봉  202501
21293  7호선   상봉  202502
20676  7호선   상봉  202503
20032  7호선   상봉  202504
19413  7호선   상봉  202505
18795  7호선   상봉  202506
18188  7호선   상봉  202507
17549  7호선   상봉  202508
16943  7호선   상봉  202509
16304  7호선   상봉  202510
15682  7호선   상봉  202511
15079  7호선   상봉  202512


In [161]:
# 중복된역(환승역)나누기
transfer_stations = seoul_mean_df[seoul_mean_df['지하철역'].duplicated(keep=False)]['지하철역'].unique()

In [162]:
# 환승역
transfer_df = seoul_mean_df[seoul_mean_df['지하철역'].isin(transfer_stations)]
transfer_df

,호선명,지하철역,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중
0,1호선,동대문,363678.0,125523.0,356249.0,125026.0,719927.0,250549.0,34.80
1,1호선,동묘앞,308894.0,157888.0,316892.0,159511.0,625786.0,317399.0,50.72
2,1호선,서울역,1794960.0,229681.0,1701907.0,217787.0,3496867.0,447468.0,12.80
3,1호선,시청,762366.0,107582.0,766076.0,105764.0,1528442.0,213346.0,13.96
4,1호선,신설동,411861.0,120797.0,397175.0,117111.0,809036.0,237908.0,29.41
...,...,...,...,...,...,...,...,...,...
290,9호선,고속터미널,469482.0,72545.0,644629.0,86716.0,1114111.0,159261.0,14.29
294,9호선,김포공항,207590.0,20139.0,322049.0,27599.0,529639.0,47738.0,9.01
297,9호선,당산,557799.0,73618.0,548013.0,68347.0,1105812.0,141965.0,12.84
298,9호선,동작(현충원),60162.0,15160.0,52065.0,13729.0,112227.0,28889.0,25.74


In [163]:
# 비환승역
non_transfer_df = seoul_mean_df[~seoul_mean_df['지하철역'].isin(transfer_stations)]
non_transfer_df

,호선명,지하철역,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중
5,1호선,제기동,501317.0,269661.0,511782.0,290055.0,1013099.0,559716.0,55.25
6,1호선,종각,1112527.0,151760.0,1083408.0,140929.0,2195935.0,292689.0,13.33
8,1호선,종로5가,712905.0,247967.0,697910.0,240352.0,1410815.0,488319.0,34.61
9,1호선,청량리(서울시립대입구),662997.0,280334.0,656727.0,281655.0,1319724.0,561989.0,42.58
10,2호선,강남,2302759.0,161221.0,2246564.0,140970.0,4549323.0,302191.0,6.64
...,...,...,...,...,...,...,...,...,...
307,9호선,신방화,202870.0,49967.0,198595.0,50170.0,401465.0,100137.0,24.94
308,9호선,양천향교,289659.0,49752.0,294136.0,49308.0,583795.0,99060.0,16.97
310,9호선,염창,514529.0,80870.0,492565.0,78794.0,1007094.0,159664.0,15.85
311,9호선,증미,219073.0,38743.0,210572.0,38823.0,429645.0,77566.0,18.05


In [164]:
# 환승역 합치기.
transfer_grouped = transfer_df.groupby('지하철역')[['총승차인원', '무임승차인원', '총하차인원', '무임하차인원', '전체승하차', '무임승하차']].sum().round(0).reset_index()

# 비중 재계산
transfer_grouped['무임승하차비중'] = (transfer_grouped['무임승하차'] / transfer_grouped['전체승하차'] * 100).round(2)

# 역명에 (환승) 태그 추가
transfer_grouped['지하철역'] = transfer_grouped['지하철역'] + '(환승)'

print(transfer_grouped)
transfer_grouped.head()

                  지하철역      총승차인원    무임승차인원      총하차인원    무임하차인원      전체승하차  \
0             가락시장(환승)   499142.0  149493.0   517546.0  148748.0  1016688.0   
1             건대입구(환승)  1514545.0  160449.0  1584989.0  158811.0  3099534.0   
2            고속터미널(환승)  2526704.0  378399.0  2523557.0  369982.0  5050261.0   
3               공덕(환승)   948702.0  133694.0   949611.0  125787.0  1898313.0   
4       교대(법원.검찰청)(환승)  1330717.0  237048.0  1305183.0  227756.0  2635900.0   
5           군자(능동)(환승)   785877.0  132473.0   726097.0  123065.0  1511974.0   
6             김포공항(환승)   530431.0   70323.0   557489.0   71426.0  1087920.0   
7               노원(환승)  1278860.0  265098.0  1389731.0  265470.0  2668591.0   
8               당산(환승)  1086434.0  134157.0  1153742.0  134381.0  2240176.0   
9         대림(구로구청)(환승)   989147.0  182965.0  1006134.0  186565.0  1995281.0   
10             동대문(환승)   932876.0  279178.0   940024.0  274204.0  1872900.0   
11  동대문역사문화공원(DDP)(환승)  1099733.0  133930.0  1106872

,지하철역,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중
0,가락시장(환승),499142.0,149493.0,517546.0,148748.0,1016688.0,298241.0,29.33
1,건대입구(환승),1514545.0,160449.0,1584989.0,158811.0,3099534.0,319260.0,10.30
2,고속터미널(환승),2526704.0,378399.0,2523557.0,369982.0,5050261.0,748381.0,14.82
3,공덕(환승),948702.0,133694.0,949611.0,125787.0,1898313.0,259481.0,13.67
4,교대(법원.검찰청)(환승),1330717.0,237048.0,1305183.0,227756.0,2635900.0,464804.0,17.63


In [165]:
transfer_grouped['호선명'] = '환승'

In [166]:
df_all = pd.concat([non_transfer_df, transfer_grouped], ignore_index=True)
df_all

,호선명,지하철역,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중
0,1호선,제기동,501317.0,269661.0,511782.0,290055.0,1013099.0,559716.0,55.25
1,1호선,종각,1112527.0,151760.0,1083408.0,140929.0,2195935.0,292689.0,13.33
2,1호선,종로5가,712905.0,247967.0,697910.0,240352.0,1410815.0,488319.0,34.61
3,1호선,청량리(서울시립대입구),662997.0,280334.0,656727.0,281655.0,1319724.0,561989.0,42.58
4,2호선,강남,2302759.0,161221.0,2246564.0,140970.0,4549323.0,302191.0,6.64
...,...,...,...,...,...,...,...,...,...
245,환승,청구(환승),219899.0,44565.0,211920.0,42532.0,431819.0,87097.0,20.17
246,환승,충무로(환승),867520.0,119883.0,893846.0,121889.0,1761366.0,241772.0,13.73
247,환승,충정로(경기대입구)(환승),417744.0,63616.0,436275.0,64124.0,854019.0,127740.0,14.96
248,환승,태릉입구(환승),438855.0,87568.0,443702.0,85169.0,882557.0,172737.0,19.57


In [167]:
df_all['호선명'] = df_all['호선명'].str.replace('호선', '').str.strip()

print(df_all['호선명'].unique())

['1' '2' '3' '4' '5' '6' '7' '8' '9' '환승']


In [168]:
df_all['지하철역'] = df_all['지하철역'].str.replace(r'\(.*\)', '', regex=True).str.strip()

In [169]:
df_all['지하철역']

0       제기동
1        종각
2      종로5가
3       청량리
4        강남
       ... 
245      청구
246     충무로
247     충정로
248    태릉입구
249      합정
Name: 지하철역, Length: 250, dtype: object

In [170]:
df_all

,호선명,지하철역,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중
0,1,제기동,501317.0,269661.0,511782.0,290055.0,1013099.0,559716.0,55.25
1,1,종각,1112527.0,151760.0,1083408.0,140929.0,2195935.0,292689.0,13.33
2,1,종로5가,712905.0,247967.0,697910.0,240352.0,1410815.0,488319.0,34.61
3,1,청량리,662997.0,280334.0,656727.0,281655.0,1319724.0,561989.0,42.58
4,2,강남,2302759.0,161221.0,2246564.0,140970.0,4549323.0,302191.0,6.64
...,...,...,...,...,...,...,...,...,...
245,환승,청구,219899.0,44565.0,211920.0,42532.0,431819.0,87097.0,20.17
246,환승,충무로,867520.0,119883.0,893846.0,121889.0,1761366.0,241772.0,13.73
247,환승,충정로,417744.0,63616.0,436275.0,64124.0,854019.0,127740.0,14.96
248,환승,태릉입구,438855.0,87568.0,443702.0,85169.0,882557.0,172737.0,19.57


In [171]:
# 주요 환승역 매핑 정보를 구성
transfer_mapping = {
    '가락시장': '3,8', '건대입구': '2,7', '고속터미널': '3,7,9', '공덕': '5,6',
    '교대': '2,3', '군자': '5,7', '김포공항': '5,9', '노원': '4,7',
    '당산': '2,9', '대림': '2,7', '동대문': '1,4', '동대문역사문화공원': '2,4,5',
    '동묘앞': '1,6', '동작': '4,9', '불광': '3,6', '사당': '2,4',
    '삼각지': '4,6', '서울역': '1,4', '시청': '1,2', '신당': '2,6',
    '신설동': '1,2', '약수': '3,6', '여의도': '5,9', '연신내': '3,6',
    '영등포구청': '2,5', '오금': '3,5', '왕십리': '2,5', '을지로3가': '2,3',
    '을지로4가': '2,5', '잠실': '2,8', '종로3가': '1,3,5', '천호': '5,8',
    '청구': '5,6', '충무로': '3,4', '충정로': '2,5', '태릉입구': '6,7', '합정': '2,6'
}

# '호선명'이 '환승'인 경우에만 딕셔너리에서 역명을 찾아 호선 정보로 교체
def update_line_name(row):
    if row['호선명'] == '환승':
        return transfer_mapping.get(row['지하철역'], row['호선명'])
    return row['호선명']

In [172]:
# apply 함수를 사용해 한 번에 적용
df_all['호선명'] = df_all.apply(update_line_name, axis=1)

In [173]:
df_all.tail()

,호선명,지하철역,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중
245,"5,6",청구,219899.0,44565.0,211920.0,42532.0,431819.0,87097.0,20.17
246,"3,4",충무로,867520.0,119883.0,893846.0,121889.0,1761366.0,241772.0,13.73
247,"2,5",충정로,417744.0,63616.0,436275.0,64124.0,854019.0,127740.0,14.96
248,"6,7",태릉입구,438855.0,87568.0,443702.0,85169.0,882557.0,172737.0,19.57
249,"2,6",합정,1379809.0,113748.0,1461301.0,111461.0,2841110.0,225209.0,7.93


In [174]:
df_all.shape

(250, 9)

In [175]:
# 중복 데이터 있는지 확인
df_all[df_all.duplicated(subset='지하철역')]

,호선명,지하철역,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중
229,"4,6",삼각지,452326.0,46910.0,462092.0,44925.0,914418.0,91835.0,10.04


In [176]:
# 삼각지역이 중복 데이터임을 확인 가능
df_all[df_all['지하철역'] == '삼각지']

,호선명,지하철역,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중
228,"4,6",삼각지,408593.0,45762.0,409492.0,43629.0,818085.0,89391.0,10.93
229,"4,6",삼각지,452326.0,46910.0,462092.0,44925.0,914418.0,91835.0,10.04


In [177]:
# groupby할 때 sort=False를 주면 기존 데이터 순서를 최대한 유지합니다.
df_all_clean = df_all.groupby('지하철역', sort=False).agg({
    '호선명': 'first',
    '총승차인원': 'sum',
    '무임승차인원': 'sum',
    '총하차인원': 'sum',
    '무임하차인원': 'sum',
    '전체승하차': 'sum',
    '무임승하차': 'sum',
    '무임승하차비중': 'sum'
}).reset_index()

# 결과 확인
# 이제 삼각지가 아까 그 근처 순서에 그대로 있는지 확인해보세요!
display(df_all_clean[df_all_clean['지하철역'] == '삼각지'])
print("최종 Shape:", df_all_clean.shape) # (249, 9)

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중
228,삼각지,"4,6",860919.0,92672.0,871584.0,88554.0,1732503.0,181226.0,20.97


최종 Shape: (249, 9)


In [178]:
# 합쳐진 데이터에서 삼각지만 쏙 뽑아서 확인해보기
df_all_clean[df_all_clean['지하철역'] == '삼각지']

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중
228,삼각지,"4,6",860919.0,92672.0,871584.0,88554.0,1732503.0,181226.0,20.97


In [96]:
# df_all.to_csv('01_passenger.csv', index=False, encoding='utf-8-sig')

# 편의시설 데이터 전처리

In [97]:
# 1-8호선 데이터셋 불러오기
line_1to8 = pd.read_csv("../../data/raw/서울교통공사_1-8호선_승강기_가동현황.csv", encoding='cp949')

In [98]:
line_1to8.head()

,역코드,역명,승강기명,운행구간,설치위치,운행상태,승강기 구분
0,2723,면목,승강기)에스컬레이터-면목 내부 2호기,B3-B1,대합실,사용가능,ES
1,2723,면목,승강기)엘리베이터-면목 내부1,B1-B3-B4,상봉 방면4-4,사용가능,EV
2,2723,면목,승강기)엘리베이터-면목 내부2,B3-B4,사가정 방면5-2,사용가능,EV
3,2723,면목,승강기)엘리베이터-면목 외부3,B1-1F,3번 출입구,사용가능,EV
4,150,서울역(1),승강기)에스컬레이터-서울(1)역 4번출구 4호기,B1-1F,4번 출입구,사용가능,ES


In [102]:
# 특정 경고(UserWarning)만 무시하도록 설정
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

# 9호선 엘레베이터, 에스컬레이터, 무빙워크 데이터셋 불러오기
line9_es = pd.read_excel("../../data/raw/서울메트로9호선_에스컬레이터_20250103.xlsx", header=3)
line9_ev = pd.read_excel("../../data/raw/서울메트로9호선_엘리베이터_20250103.xlsx", header=3)
line9_mw = pd.read_excel("../../data/raw/서울메트로9호선_무빙워크_20250103.xlsx", header=3)

In [103]:
line9_es.head(1)

,No,철도운영기관명,운영노선명,역명,관리번호,상하행구분,(근접)출입구번호,시작층(지상/지하),시작층(운행역층),시작층(상세위치),종료층(지상/지하),종료층(운행역층),종료층(상세위치),승강기 상태,승강기형폭,승강기 일련번호,데이터 기준일자,참고사항
0,1,서울9호선,9호선,개화역,1,상행,NaN,지상,1,(1F) 김포공항역 방향 승강장 6-4 출입문 앞,지상,2,(2F) 표 내는 곳 앞,운행,1200형,1804-297,20231218,NaN


In [104]:
line9_ev.head(1)

,No,철도운영기관명,운영노선명,역명,관리번호,(근접)출입구번호,상세위치,시작층(지상/지하),시작층(운행역층),종료층(지상/지하),종료층(운행역층),정원(인원수),정원(중량)(kg),승강기 상태,승강기 일련번호,데이터 기준일자,참고사항,Unnamed: 17
0,1,서울9호선,9호선,개화역,1,NaN,(1F) 하선승강장 6-4 근처\n(2F) 안전관리실 옆,지상,1,지상,2,15.0,1000,운행,0029-358,20230508,NaN,NaN


In [105]:
line9_mw.head(1)

,No,철도운영기관명,운영노선명,역명,관리번호,지상지하구분,역층,(근접) 출입구번호,시작(운행방향),시작(상세위치),종료(운행방향),종료(상세위치),승강기상태코드,승강기 일련번호,데이터 기준일자,참고사항
0,1,서울9호선,9호선,동작,1,지하,2,NaN,NaN,(B2) 9호선 환승통로 앞,NaN,(B2) 9호선 환승에스컬레이터 앞,운행,1804-622,20231219,NaN


In [106]:
# '승강기명' 컬럼 통일 (시작위치 + 관리번호)
line9_es['관리번호'] = line9_es['관리번호'].astype(str)
line9_es['승강기명'] = line9_es['시작층(상세위치)'] + " " + line9_es['관리번호'] + "호기"

line9_ev['관리번호'] = line9_ev['관리번호'].astype(str)            
line9_ev['승강기명'] = line9_ev['상세위치'] + " " + line9_ev['관리번호'] + "호기"

line9_mw['관리번호'] = line9_mw['관리번호'].astype(str)
line9_mw['승강기명'] = line9_mw['시작(상세위치)'] + " " + line9_mw['관리번호'] + "호기"

In [107]:
# 승강기 구분 컬럼 생성
line9_es['승강기 구분'] = 'ES'
line9_ev['승강기 구분'] = 'EV'
line9_mw['승강기 구분'] = 'MW'

In [108]:
# 1-8호선 데이터에 '운영노선명' 컬럼 생성
line_1to8['운영노선명']=''

In [109]:
# 공통 컬럼 추출하기
line9_es.columns.intersection(line9_ev.columns).intersection(line9_mw.columns)
common_cols = ['운영노선명', '역명', '승강기명', '승강기 구분']

In [111]:
# 1-8호선 데이터셋과 9호선 데이터셋 병합하기
master_df = pd.concat([line9_ev[common_cols], line9_es[common_cols], line9_mw[common_cols], line_1to8[common_cols]])
master_df.loc[master_df['역명']=='개화역', '역명'] = '개화'

In [112]:
master_df

,운영노선명,역명,승강기명,승강기 구분
0,9호선,개화,(1F) 하선승강장 6-4 근처\n(2F) 안전관리실 옆 1호기,EV
1,9호선,개화,(1F) 2번출구 옆\n(2F) 편의점 앞 2호기,EV
2,9호선,김포공항,(B1) 안전관리실 옆\n(B3) 상부본선 승강장 4-2 지점 2호기,EV
3,9호선,김포공항,(B1) 안전관리실 앞\n(B4) 하선 승강장 1-1 지점 3호기,EV
4,9호선,김포공항,(B1) 안전관리실 앞\n(B4) 하선 승강장 1-1 지점 4호기,EV
...,...,...,...,...
2806,,남위례,승강기)에스컬레이터-남위례 내부 5호기,ES
2807,,남위례,승강기)에스컬레이터-남위례 내부 6호기,ES
2808,,남위례,승강기)엘리베이터-남위례 내부 1호기,EV
2809,,남위례,승강기)엘리베이터-남위례 내부 2호기,EV


In [113]:
# master_df.to_csv('02_facility', index=False)

# 유무임 인원과 편의시설 데이터 결합 후 전처리

In [179]:
subway_final = pd.read_csv("../../data/processed/team/01_passenger.csv")
subway_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249 entries, 0 to 248
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   지하철역     249 non-null    object 
 1   호선명      249 non-null    object 
 2   총승차인원    249 non-null    float64
 3   무임승차인원   249 non-null    float64
 4   총하차인원    249 non-null    float64
 5   무임하차인원   249 non-null    float64
 6   전체승하차    249 non-null    float64
 7   무임승하차    249 non-null    float64
 8   무임승하차비중  249 non-null    float64
dtypes: float64(7), object(2)
memory usage: 17.6+ KB


In [180]:
master_df = pd.read_csv("../../data/processed/team/02_facility.csv")
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3385 entries, 0 to 3384
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   운영노선명   574 non-null    object
 1   역명      3385 non-null   object
 2   승강기명    3385 non-null   object
 3   승강기 구분  3385 non-null   object
dtypes: object(4)
memory usage: 105.9+ KB


In [181]:
master_df.rename(columns = {'운영노선명' : '호선명', '역명' :'지하철역'}, inplace=True)
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3385 entries, 0 to 3384
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   호선명     574 non-null    object
 1   지하철역    3385 non-null   object
 2   승강기명    3385 non-null   object
 3   승강기 구분  3385 non-null   object
dtypes: object(4)
memory usage: 105.9+ KB


In [182]:
line_map = subway_final.set_index('지하철역')['호선명']
line_map

지하철역
제기동       1
종각        1
종로5가      1
청량리       1
강남        2
       ... 
청구      5,6
충무로     3,4
충정로     2,5
태릉입구    6,7
합정      2,6
Name: 호선명, Length: 249, dtype: object

In [183]:
master_df['호선명'] = master_df['호선명'].replace('', np.nan)
master_df['호선명'] = master_df['호선명'].fillna(master_df['지하철역'].map(line_map))

In [184]:
master_df.loc[master_df['호선명'] == '9호선', '호선명'] = "9"
master_df[master_df['호선명'].isna()]

,호선명,지하철역,승강기명,승강기 구분
578,NaN,서울역(1),승강기)에스컬레이터-서울(1)역 4번출구 4호기,ES
579,NaN,서울역(1),승강기)에스컬레이터-서울(1)역 4번출구 5호기,ES
580,NaN,서울역(1),승강기)에스컬레이터-서울(1)역 상행(9-3) 3호기,ES
581,NaN,서울역(1),승강기)에스컬레이터-서울(1)역 하행(2-3) 2호기,ES
582,NaN,서울역(1),승강기)에스컬레이터-서울역(1) 4호선연결통로 1호기,ES
...,...,...,...,...
3380,NaN,남위례,승강기)에스컬레이터-남위례 내부 5호기,ES
3381,NaN,남위례,승강기)에스컬레이터-남위례 내부 6호기,ES
3382,NaN,남위례,승강기)엘리베이터-남위례 내부 1호기,EV
3383,NaN,남위례,승강기)엘리베이터-남위례 내부 2호기,EV


In [185]:
master_df['호선명'] = master_df['호선명'].fillna(master_df['지하철역'].str.split("(").str[1].str.replace(")", ""))
master_df['호선명'].unique()

array(['9', '7', '1', '2', '3', '4', '전쟁기념관', '6', '5', nan, '뚝섬한강공원',
       '8'], dtype=object)

In [186]:
master_df.loc[master_df['호선명'] == '전쟁기념관', '호선명'] = "4"
master_df.loc[master_df['호선명'] == '뚝섬한강공원', '호선명'] = "7"
master_df['호선명'].unique()

array(['9', '7', '1', '2', '3', '4', '6', '5', nan, '8'], dtype=object)

In [187]:
master_df['지하철역'] = master_df['지하철역'].str.split("(").str[0]
master_df

,호선명,지하철역,승강기명,승강기 구분
0,9,개화,(1F) 하선승강장 6-4 근처\n(2F) 안전관리실 옆 1호기,EV
1,9,개화,(1F) 2번출구 옆\n(2F) 편의점 앞 2호기,EV
2,9,김포공항,(B1) 안전관리실 옆\n(B3) 상부본선 승강장 4-2 지점 2호기,EV
3,9,김포공항,(B1) 안전관리실 앞\n(B4) 하선 승강장 1-1 지점 3호기,EV
4,9,김포공항,(B1) 안전관리실 앞\n(B4) 하선 승강장 1-1 지점 4호기,EV
...,...,...,...,...
3380,NaN,남위례,승강기)에스컬레이터-남위례 내부 5호기,ES
3381,NaN,남위례,승강기)에스컬레이터-남위례 내부 6호기,ES
3382,NaN,남위례,승강기)엘리베이터-남위례 내부 1호기,EV
3383,NaN,남위례,승강기)엘리베이터-남위례 내부 2호기,EV


In [188]:
# 서울 시내에 있는 역이 아닌 행 삭제
non_seoul = [
    '미사', '하남검단산', '하남시청', '하남풍산',
    '광명사거리', '굴포천', '까치울', '부천시청', '부평구청',
    '삼산체육관', '상동', '신중동', '춘의', '철산', 
    '남위례', '남한산성입구', '단대오거리',
    '모란', '복정', '산성', '수진', '신흥'
]
master_df = master_df[~master_df['지하철역'].isin(non_seoul)]

In [189]:
master_df[~master_df['호선명'].isin(["1", "2", "3", "4", "5", "6", "7", "8", "9"])]['호선명'].unique()

array([], dtype=object)

In [190]:
# '지하철역' 과 '승강기 구분' 기준으로 개수 집계하기
master_pivot = master_df.pivot_table(
                                index=['지하철역'],
                                columns='승강기 구분',
                                aggfunc='size',
                                fill_value=0
                                ).reset_index() 

In [191]:
# subway_final과 master_df 병합하기
merged = pd.merge(subway_final, master_pivot, on='지하철역', how='left')
merged

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,ES,EV,MW,WL
0,제기동,1,501317.0,269661.0,511782.0,290055.0,1013099.0,559716.0,55.25,2.0,3.0,0.0,0.0
1,종각,1,1112527.0,151760.0,1083408.0,140929.0,2195935.0,292689.0,13.33,2.0,4.0,0.0,0.0
2,종로5가,1,712905.0,247967.0,697910.0,240352.0,1410815.0,488319.0,34.61,0.0,3.0,0.0,0.0
3,청량리,1,662997.0,280334.0,656727.0,281655.0,1319724.0,561989.0,42.58,6.0,2.0,0.0,1.0
4,강남,2,2302759.0,161221.0,2246564.0,140970.0,4549323.0,302191.0,6.64,0.0,4.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
244,청구,"5,6",219899.0,44565.0,211920.0,42532.0,431819.0,87097.0,20.17,10.0,5.0,0.0,0.0
245,충무로,"3,4",867520.0,119883.0,893846.0,121889.0,1761366.0,241772.0,13.73,23.0,4.0,0.0,0.0
246,충정로,"2,5",417744.0,63616.0,436275.0,64124.0,854019.0,127740.0,14.96,8.0,4.0,0.0,0.0
247,태릉입구,"6,7",438855.0,87568.0,443702.0,85169.0,882557.0,172737.0,19.57,24.0,6.0,0.0,0.0


In [192]:
merged.loc[merged['EV'] == 0, '지하철역']

46     도곡
201    사평
Name: 지하철역, dtype: object

In [193]:
# 엘리베이터 수 조정하기
merged.loc[merged['지하철역']=='도곡', 'EV'] = 2
merged.loc[merged['지하철역']=='사평', 'EV'] = 4

In [194]:
# 개명된 역 삭제하기
drop_stations = ['뚝섬유원지', '당고개', '신내']
merged = merged[~merged['지하철역'].isin(drop_stations)]

In [195]:
merged = merged.copy()

# ES + EV 편의시설의 개수를 집계하는 'TOTAL' 컬럼 생성하기
merged['TOTAL'] = merged['ES'] + merged['EV']
merged[['ES', 'EV', 'TOTAL']] = merged[['ES', 'EV', 'TOTAL']].astype('Int64')
merged

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,ES,EV,MW,WL,TOTAL
0,제기동,1,501317.0,269661.0,511782.0,290055.0,1013099.0,559716.0,55.25,2,3,0.0,0.0,5
1,종각,1,1112527.0,151760.0,1083408.0,140929.0,2195935.0,292689.0,13.33,2,4,0.0,0.0,6
2,종로5가,1,712905.0,247967.0,697910.0,240352.0,1410815.0,488319.0,34.61,0,3,0.0,0.0,3
3,청량리,1,662997.0,280334.0,656727.0,281655.0,1319724.0,561989.0,42.58,6,2,0.0,1.0,8
4,강남,2,2302759.0,161221.0,2246564.0,140970.0,4549323.0,302191.0,6.64,0,4,0.0,0.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
244,청구,"5,6",219899.0,44565.0,211920.0,42532.0,431819.0,87097.0,20.17,10,5,0.0,0.0,15
245,충무로,"3,4",867520.0,119883.0,893846.0,121889.0,1761366.0,241772.0,13.73,23,4,0.0,0.0,27
246,충정로,"2,5",417744.0,63616.0,436275.0,64124.0,854019.0,127740.0,14.96,8,4,0.0,0.0,12
247,태릉입구,"6,7",438855.0,87568.0,443702.0,85169.0,882557.0,172737.0,19.57,24,6,0.0,0.0,30


In [196]:
merged = merged.drop(['MW', 'WL'], axis=1)
merged

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,ES,EV,TOTAL
0,제기동,1,501317.0,269661.0,511782.0,290055.0,1013099.0,559716.0,55.25,2,3,5
1,종각,1,1112527.0,151760.0,1083408.0,140929.0,2195935.0,292689.0,13.33,2,4,6
2,종로5가,1,712905.0,247967.0,697910.0,240352.0,1410815.0,488319.0,34.61,0,3,3
3,청량리,1,662997.0,280334.0,656727.0,281655.0,1319724.0,561989.0,42.58,6,2,8
4,강남,2,2302759.0,161221.0,2246564.0,140970.0,4549323.0,302191.0,6.64,0,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...
244,청구,"5,6",219899.0,44565.0,211920.0,42532.0,431819.0,87097.0,20.17,10,5,15
245,충무로,"3,4",867520.0,119883.0,893846.0,121889.0,1761366.0,241772.0,13.73,23,4,27
246,충정로,"2,5",417744.0,63616.0,436275.0,64124.0,854019.0,127740.0,14.96,8,4,12
247,태릉입구,"6,7",438855.0,87568.0,443702.0,85169.0,882557.0,172737.0,19.57,24,6,30


In [197]:
# 서울역 이름 서울로 변경
merged['지하철역'] = merged['지하철역'].replace('서울역', '서울')

In [198]:
merged.shape

(246, 12)

In [199]:
merged[merged['지하철역'].isin(['이수', '총신대입구'])]

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,ES,EV,TOTAL
77,총신대입구,4,564012.0,159811.0,610462.0,167328.0,1174474.0,327139.0,27.85,4,4,8
173,이수,7,479384.0,85118.0,463584.0,80754.0,942968.0,165872.0,17.59,23,3,26


In [200]:
# 이수역과 총신대입구역을 합치는 코드
isu_df = merged[merged['지하철역'].isin(['이수', '총신대입구'])]

new_isu_row = {
    '지하철역': '이수',
    '호선명': '4,7',
    '총승차인원': isu_df['총승차인원'].sum(),
    '무임승차인원': isu_df['무임승차인원'].sum(),
    '총하차인원': isu_df['총하차인원'].sum(),
    '무임하차인원': isu_df['무임하차인원'].sum(),
    '전체승하차': isu_df['전체승하차'].sum(),
    '무임승하차': isu_df['무임승하차'].sum(),
    'ES': isu_df['ES'].sum(),
    'EV': isu_df['EV'].sum(),
    'TOTAL': isu_df['TOTAL'].sum()
}

new_isu_row['무임승하차비중'] = round((new_isu_row['무임승하차'] / new_isu_row['전체승하차']) * 100, 2)

merged = merged[~merged['지하철역'].isin(['이수', '총신대입구'])].copy()
merged = pd.concat([merged, pd.DataFrame([new_isu_row])], ignore_index=True)

merged

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,ES,EV,TOTAL
0,제기동,1,501317.0,269661.0,511782.0,290055.0,1013099.0,559716.0,55.25,2,3,5
1,종각,1,1112527.0,151760.0,1083408.0,140929.0,2195935.0,292689.0,13.33,2,4,6
2,종로5가,1,712905.0,247967.0,697910.0,240352.0,1410815.0,488319.0,34.61,0,3,3
3,청량리,1,662997.0,280334.0,656727.0,281655.0,1319724.0,561989.0,42.58,6,2,8
4,강남,2,2302759.0,161221.0,2246564.0,140970.0,4549323.0,302191.0,6.64,0,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...
240,충무로,"3,4",867520.0,119883.0,893846.0,121889.0,1761366.0,241772.0,13.73,23,4,27
241,충정로,"2,5",417744.0,63616.0,436275.0,64124.0,854019.0,127740.0,14.96,8,4,12
242,태릉입구,"6,7",438855.0,87568.0,443702.0,85169.0,882557.0,172737.0,19.57,24,6,30
243,합정,"2,6",1379809.0,113748.0,1461301.0,111461.0,2841110.0,225209.0,7.93,16,7,23


In [201]:
# 인원수 정수화
target_cols = ['총승차인원', '무임승차인원', '총하차인원', '무임하차인원', '전체승하차', '무임승하차']
merged[target_cols] = merged[target_cols].astype(int)
merged

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,ES,EV,TOTAL
0,제기동,1,501317,269661,511782,290055,1013099,559716,55.25,2,3,5
1,종각,1,1112527,151760,1083408,140929,2195935,292689,13.33,2,4,6
2,종로5가,1,712905,247967,697910,240352,1410815,488319,34.61,0,3,3
3,청량리,1,662997,280334,656727,281655,1319724,561989,42.58,6,2,8
4,강남,2,2302759,161221,2246564,140970,4549323,302191,6.64,0,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...
240,충무로,"3,4",867520,119883,893846,121889,1761366,241772,13.73,23,4,27
241,충정로,"2,5",417744,63616,436275,64124,854019,127740,14.96,8,4,12
242,태릉입구,"6,7",438855,87568,443702,85169,882557,172737,19.57,24,6,30
243,합정,"2,6",1379809,113748,1461301,111461,2841110,225209,7.93,16,7,23


In [202]:
# 지축역은 서울시 경계와 애매하기 때문에 삭제
merged = merged[~(merged['지하철역'] == '지축')]
merged

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,ES,EV,TOTAL
0,제기동,1,501317,269661,511782,290055,1013099,559716,55.25,2,3,5
1,종각,1,1112527,151760,1083408,140929,2195935,292689,13.33,2,4,6
2,종로5가,1,712905,247967,697910,240352,1410815,488319,34.61,0,3,3
3,청량리,1,662997,280334,656727,281655,1319724,561989,42.58,6,2,8
4,강남,2,2302759,161221,2246564,140970,4549323,302191,6.64,0,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...
240,충무로,"3,4",867520,119883,893846,121889,1761366,241772,13.73,23,4,27
241,충정로,"2,5",417744,63616,436275,64124,854019,127740,14.96,8,4,12
242,태릉입구,"6,7",438855,87568,443702,85169,882557,172737,19.57,24,6,30
243,합정,"2,6",1379809,113748,1461301,111461,2841110,225209,7.93,16,7,23


In [203]:
# 가중치를 적용하여 시설부하지수 컬럼 생성
merged = merged[['지하철역', '호선명', '총승차인원', '무임승차인원', '총하차인원', '무임하차인원', '전체승하차', '무임승하차', '무임승하차비중', 'EV', 'ES', 'TOTAL']].copy()
merged['시설부하지수'] = merged['무임승하차'] / (merged['EV'] * 4 + merged['ES'] * 1)
merged

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,EV,ES,TOTAL,시설부하지수
0,제기동,1,501317,269661,511782,290055,1013099,559716,55.25,3,2,5,39979.714286
1,종각,1,1112527,151760,1083408,140929,2195935,292689,13.33,4,2,6,16260.5
2,종로5가,1,712905,247967,697910,240352,1410815,488319,34.61,3,0,3,40693.25
3,청량리,1,662997,280334,656727,281655,1319724,561989,42.58,2,6,8,40142.071429
4,강남,2,2302759,161221,2246564,140970,4549323,302191,6.64,4,0,4,18886.9375
...,...,...,...,...,...,...,...,...,...,...,...,...,...
240,충무로,"3,4",867520,119883,893846,121889,1761366,241772,13.73,4,23,27,6199.282051
241,충정로,"2,5",417744,63616,436275,64124,854019,127740,14.96,4,8,12,5322.5
242,태릉입구,"6,7",438855,87568,443702,85169,882557,172737,19.57,6,24,30,3598.6875
243,합정,"2,6",1379809,113748,1461301,111461,2841110,225209,7.93,7,16,23,5118.386364


In [204]:
# 시설부하지수 소수점 둘째까리에서 반올림
merged['시설부하지수'] = round(merged['시설부하지수'], 2)
merged

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,EV,ES,TOTAL,시설부하지수
0,제기동,1,501317,269661,511782,290055,1013099,559716,55.25,3,2,5,39979.71
1,종각,1,1112527,151760,1083408,140929,2195935,292689,13.33,4,2,6,16260.5
2,종로5가,1,712905,247967,697910,240352,1410815,488319,34.61,3,0,3,40693.25
3,청량리,1,662997,280334,656727,281655,1319724,561989,42.58,2,6,8,40142.07
4,강남,2,2302759,161221,2246564,140970,4549323,302191,6.64,4,0,4,18886.94
...,...,...,...,...,...,...,...,...,...,...,...,...,...
240,충무로,"3,4",867520,119883,893846,121889,1761366,241772,13.73,4,23,27,6199.28
241,충정로,"2,5",417744,63616,436275,64124,854019,127740,14.96,4,8,12,5322.5
242,태릉입구,"6,7",438855,87568,443702,85169,882557,172737,19.57,6,24,30,3598.69
243,합정,"2,6",1379809,113748,1461301,111461,2841110,225209,7.93,7,16,23,5118.39


In [205]:
# merged.to_excel("subway_merged_base.xlsx", index=False)
# merged.to_csv("subway_merged_base.csv", index=False)